# 02 - Document Classification: TF-IDF+LogReg baseline vs fine-tuned DistilBERT

4-class subset of RVL-CDIP (`invoice`, `letter`, `form`, `email`) from the
`vaclavpechtor/rvl_cdip-small-200` Hugging Face dataset (200 images/class
of the full 400K-image RVL-CDIP - subsetted honestly per `prompt.md`'s
"what not to do": no full-dataset training claims here).

Both models classify on **OCR text** (Tesseract, per notebook 01's
conclusion, EasyOCR fallback on near-empty output), not the raw image -
this is a document-AI text-classification task, not image classification.
OCR text was pre-extracted by `../tmp/ocr_cache_classification.py` into
`data/processed/classification_ocr_text.csv` (one row per image: id,
split, label, text, ocr_engine, ocr_time_s).

In [1]:
# Document classification library code (inlined from src/classify.py so
# this notebook is self-contained - no dependency on ../src at runtime).
#
# TF-IDF+LogReg baseline vs fine-tuned DistilBERT, both operating on OCR
# text (not the raw image).

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline


def train_tfidf_logreg(train_texts: list[str], train_labels: list[str]) -> Pipeline:
    pipe = Pipeline(
        [
            ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2)),
            ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
        ]
    )
    pipe.fit(train_texts, train_labels)
    return pipe


def evaluate(y_true: list[str], y_pred: list[str]) -> str:
    return classification_report(y_true, y_pred, digits=3, zero_division=0)


class DistilBertTextClassifier:
    """Thin wrapper around a fine-tuned DistilBERT sequence classifier."""

    def __init__(self, model_dir: str):
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        self.model.eval()

    def predict(self, texts: list[str], batch_size: int = 16) -> list[str]:
        return [label for label, _ in self.predict_with_confidence(texts, batch_size)]

    def predict_with_confidence(self, texts: list[str], batch_size: int = 16) -> list[tuple[str, float]]:
        import torch

        id2label = self.model.config.id2label
        results = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = self.tokenizer(batch, truncation=True, padding=True, max_length=256, return_tensors="pt")
            with torch.no_grad():
                probs = torch.softmax(self.model(**enc).logits, dim=-1)
            conf, idx = probs.max(dim=-1)
            results.extend((id2label[int(i)], float(c)) for i, c in zip(idx.tolist(), conf.tolist()))
        return results


def fine_tune_distilbert(
    train_texts: list[str],
    train_labels: list[int],
    eval_texts: list[str],
    eval_labels: list[int],
    label_names: list[str],
    output_dir: str,
    epochs: int = 4,
):
    from datasets import Dataset
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        DataCollatorWithPadding,
        EarlyStoppingCallback,
        Trainer,
        TrainingArguments,
    )

    model_name = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    id2label = {i: name for i, name in enumerate(label_names)}
    label2id = {name: i for i, name in enumerate(label_names)}
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(label_names), id2label=id2label, label2id=label2id
    )

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=256)

    train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels}).map(tokenize, batched=True)
    eval_ds = Dataset.from_dict({"text": eval_texts, "label": eval_labels}).map(tokenize, batched=True)

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        learning_rate=3e-5,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=10,
        report_to=[],
    )

    def compute_metrics(eval_pred):
        from sklearn.metrics import accuracy_score, f1_score

        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro"),
        }

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainer.train()
    metrics = trainer.evaluate()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    return trainer, metrics

In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/classification_ocr_text.csv")
df["text"] = df["text"].fillna("")
print(df.groupby(["split", "label"]).size())
print()
print("OCR engine usage:")
print(df.ocr_engine.value_counts())
print()
print("rows with near-empty OCR text (<20 chars):", (df.text.str.len() < 20).sum())

split       label  
train       email      160
            form       160
            invoice    160
            letter     160
validation  email       40
            form        40
            invoice     40
            letter      40
dtype: int64

OCR engine usage:
ocr_engine
tesseract    790
easyocr       10
Name: count, dtype: int64

rows with near-empty OCR text (<20 chars): 3


In [3]:
train_df = df[df.split == "train"].reset_index(drop=True)
val_df = df[df.split == "validation"].reset_index(drop=True)

labels_sorted = sorted(train_df.label.unique())
label2id = {l: i for i, l in enumerate(labels_sorted)}
print(f"train: {len(train_df)}, val: {len(val_df)}, classes: {labels_sorted}")

train: 640, val: 160, classes: ['email', 'form', 'invoice', 'letter']


## Baseline: TF-IDF + Logistic Regression

In [4]:
baseline = train_tfidf_logreg(train_df.text.tolist(), train_df.label.tolist())
baseline_preds = baseline.predict(val_df.text.tolist())
print(evaluate(val_df.label.tolist(), baseline_preds))

              precision    recall  f1-score   support

       email      0.971     0.850     0.907        40
        form      0.696     0.800     0.744        40
     invoice      0.737     0.700     0.718        40
      letter      0.756     0.775     0.765        40

    accuracy                          0.781       160
   macro avg      0.790     0.781     0.784       160
weighted avg      0.790     0.781     0.784       160



## Fine-tuned DistilBERT

In [5]:
DISTILBERT_OUTPUT_DIR = "../models/artifacts/distilbert-classifier"

trainer, metrics = fine_tune_distilbert(
    train_texts=train_df.text.tolist(),
    train_labels=[label2id[l] for l in train_df.label],
    eval_texts=val_df.text.tolist(),
    eval_labels=[label2id[l] for l in val_df.label],
    label_names=labels_sorted,
    output_dir=DISTILBERT_OUTPUT_DIR,
    epochs=4,
)
print(metrics)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.241587,0.652372,4,0.787500,0.790473


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.6523720026016235, 'eval_accuracy': 0.7875, 'eval_macro_f1': 0.7904734237372713}


In [6]:
clf = DistilBertTextClassifier(DISTILBERT_OUTPUT_DIR)
bert_preds = clf.predict(val_df.text.tolist())
print(evaluate(val_df.label.tolist(), bert_preds))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

              precision    recall  f1-score   support

       email      0.974     0.925     0.949        40
        form      0.615     0.800     0.696        40
     invoice      0.788     0.650     0.712        40
      letter      0.838     0.775     0.805        40

    accuracy                          0.787       160
   macro avg      0.804     0.787     0.790       160
weighted avg      0.804     0.787     0.790       160



## Summary

DistilBERT edges out the TF-IDF+LogReg baseline on the aggregate
(macro-F1 0.790 vs 0.784, accuracy 0.787 vs 0.781), but the gain is
small and not uniform across classes - early stopping (patience=2 on
`macro_f1`) ran the full 4 epochs without triggering, so this is the
converged result, not an early cutoff.

Per-class F1, baseline vs DistilBERT:

| class   | baseline | DistilBERT |
|---------|----------|------------|
| email   | 0.907    | 0.949      |
| form    | 0.744    | 0.696      |
| invoice | 0.718    | 0.712      |
| letter  | 0.765    | 0.805      |

`email` is easiest for both models (distinctive vocabulary - greetings,
sign-offs). `form` and `invoice` are the hardest for both, consistent
with the hypothesis that sparse/short OCR text with similar
layout-agnostic vocabulary makes them confusable - but the specific
prediction that `form`/`letter` would be the confusable pair doesn't
hold: DistilBERT actually regresses on `form` (0.744 → 0.696) while
improving on `letter` (0.765 → 0.805). `invoice` stays essentially flat
for both models, suggesting its errors aren't easily fixed by a bigger
text-only model - likely because invoice-vs-form/letter confusion here
is more about numeric/tabular layout than about the *words* in the OCR
text.

OCR-engine fallback rate is low (10/800 rows fell back to EasyOCR, 3
rows have near-empty text) - too small a slice to check for correlation
with which examples either model gets wrong here; that's deferred to
notebook 04's proper error attribution.